# 4. Mapping fibers between patients

Uses the UAC charts from `la_construction` to transfer the fiber field of several source patients onto one target patient's mesh. For each source patient:

1. map its fibers from its original mesh onto its UAC submeshes,
2. copy them onto the target's UAC submeshes (nearest neighbor in UAC space),
3. map them back onto the target's original mesh.

Requires `uac_submesh_data.npz` for every patient involved (run `la_construction` for each).

In [ ]:
from pathlib import Path

import numpy as np
import pyvista as pv

from uacpy import mapper
from uacpy.output import serialization, visualization

## Parameters

`target_mesh_dir` is the patient receiving the fibers; `source_patient_ids` are the patients whose fibers are mapped onto it. As the comment in the next cell notes, fibers are copied from the nearest source cell rather than averaged, which preserves the roughness of the field.

In [ ]:
target_mesh_dir = Path("../data/patient_01")
source_patient_ids = (
    "patient_02",
    "patient_03",
    "patient_04",
    "patient_05",
    "patient_06",
    "patient_07",
)
original_mesh_file_name = "mesh_with_fibers_tags.vtu"
uac_mesh_file_name = "uac_submesh_data.npz"

# Fibers are copied from the nearest source cell (no averaging), which keeps the roughness of the
# fields that statistics of a field (variance, correlation length) need.
mapped_fibers_file_name = "fiber_fields_mapped_to_target.npz"


## Load the target patient

Loads the target's original mesh, its UAC submeshes and its fibers, and creates a `UACMapper` (which caches the per-submesh Jacobians). It also writes the target's own fibers (`fiber_field.npy`) and an attribute-free copy of its mesh (`mesh.vtu`) next to the data, so the mapped fields can be used with the geometry alone.

In [ ]:
target_mesh_original = pv.read(target_mesh_dir / original_mesh_file_name)
target_mesh_uac = serialization.load_uac_submeshdata(target_mesh_dir / uac_mesh_file_name)
target_mesh_fibers = np.array(target_mesh_original.cell_data["fibers"])
target_uac_mapper = mapper.UACMapper(target_mesh_original, target_mesh_uac)

target_mesh_copy = target_mesh_original.copy()
target_mesh_copy.clear_cell_data()
target_mesh_copy.clear_point_data()
np.save(target_mesh_dir / "fiber_field.npy", target_mesh_fibers)
pv.save_meshio(target_mesh_dir / "mesh.vtu", target_mesh_copy)

## Map each source patient's fibers onto the target

For every source patient: load its mesh and UAC submeshes, move its fibers to the source UAC mesh (`map_vector_from_original_to_uac_mesh`), transfer them to the target UAC mesh (`interpolate_to_other_uac_mesh`, nearest-neighbor copy), and map them back to the target's original mesh (`map_vector_from_uac_to_original_mesh`, normalized to unit length). All mapped fields are saved together in one `.npz`, keyed by source patient.

**Note:** vectors are converted between a mesh and its UAC chart with the cell-wise Jacobians, one `(3, 2)` matrix per cell mapping `(alpha, beta)` directions to 3D tangent vectors. Going to UAC space, each cell's 3D fiber is multiplied by the pseudoinverse of that cell's Jacobian, giving a 2D vector in `(alpha, beta)`. Going back, the target cell's own Jacobian turns the 2D vector into a 3D tangent vector. Only the 2D vectors are copied between patients, so the fiber direction relative to the UAC axes is preserved, while each patient's own cell geometry is respected. The length of a mapped vector only reflects how much the chart stretches the cell, hence the normalization.

In [ ]:
fiber_fields_mapped_to_target = dict.fromkeys(source_patient_ids)
for patient_id in source_patient_ids:
    source_mesh_dir = Path(f"../data/{patient_id}")
    source_mesh_original = pv.read(source_mesh_dir / original_mesh_file_name)
    source_mesh_uac = serialization.load_uac_submeshdata(source_mesh_dir / uac_mesh_file_name)
    source_mesh_fibers = np.array(source_mesh_original.cell_data["fibers"])
    source_uac_mapper = mapper.UACMapper(source_mesh_original, source_mesh_uac)
    fibers_source_uac_mesh = source_uac_mapper.map_vector_from_original_to_uac_mesh(
        source_mesh_fibers
    )
    fibers_target_uac_mesh = source_uac_mapper.interpolate_to_other_uac_mesh(
        fibers_source_uac_mesh, target_uac_mapper
    )
    target_mesh_fibers = target_uac_mapper.map_vector_from_uac_to_original_mesh(
        fibers_target_uac_mesh, normalize=True
    )
    fiber_fields_mapped_to_target[patient_id] = target_mesh_fibers

np.savez(target_mesh_dir / mapped_fibers_file_name, **fiber_fields_mapped_to_target)

## Visualize the transfer (last source patient)

In order: the source fibers on its original mesh (blue), on its UAC mesh (blue), the transferred fibers on the target's UAC mesh (green), and finally on the target's original mesh (green). The patterns should look consistent across the four views.

In [ ]:
visualization.plot_mesh_with_vector_field(
    source_mesh_original,
    source_mesh_original.cell_data["fibers"],
    scaling_factor=1,
    vector_color="blue",
)

visualization.plot_uac_mesh_with_vector_field(
    source_mesh_uac,
    fibers_source_uac_mesh,
    scaling_factor=0.02,
    vector_color="blue",
    mesh_color="lightgray",
)

visualization.plot_uac_mesh_with_vector_field(
    target_mesh_uac,
    fibers_target_uac_mesh,
    scaling_factor=0.02,
    vector_color="green",
    mesh_color="lightgray",
)

visualization.plot_mesh_with_vector_field(
    target_mesh_original,
    target_mesh_fibers,
    scaling_factor=1,
    vector_color="green",
    mesh_color="lightgray",
)